Crearemos el primer modelo de predicción para predecir si una orden será devuelta o no, con la feature que cramos anteriormente (is_returned =1 or 0)
Esto ayudaría a la empresa a preparar devoluciones futuras

In [66]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from imblearn.over_sampling import SMOTE

df = pd.read_csv('/content/READYecommerce_raw_dataset.csv')


In [67]:
# Features and target
features = ['age', 'quantity', 'unit_price', 'discount_pct',
            'order_month', 'order_dayofweek', 'is_discounted',
            'category_enc', 'country_enc', 'payment_method_enc',
            'shipping_method_enc']

X = df[features]
y = df['is_returned']



In [68]:
# Split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)


Aplicando SMOTE antes del entrenamiento para balancear el dataset artificialmente ya que el modelo inicial no estaba balanceado porque habían muy pocos datos de la clase 0:

Clase 0 (no devuelta): 1405 filas → 93.7%
Clase 1 (devuelta):      95 filas →  6.3%

In [69]:
# Imputar valores nulos ANTES de SMOTE
imputer = SimpleImputer(strategy='median')
X_train_imputed = imputer.fit_transform(X_train)
X_test_imputed = imputer.transform(X_test)  # mismo imputer, solo transform

# Ahora sí aplicar SMOTE
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_imputed, y_train)



In [70]:
# Verificar balance
print("Antes:", pd.Series(y_train).value_counts().to_dict())
print("Después:", pd.Series(y_train_sm).value_counts().to_dict())

Antes: {0: 1124, 1: 76}
Después: {0: 1124, 1: 1124}


In [71]:
# Entrenar con datos balanceados
model = RandomForestClassifier(n_estimators=100,
                               class_weight='balanced',
                               random_state=42)
model.fit(X_train_sm, y_train_sm)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [72]:
# Evaluate
y_pred = model.predict(X_test_imputed)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.94      1.00      0.97       281
           1       0.00      0.00      0.00        19

    accuracy                           0.93       300
   macro avg       0.47      0.50      0.48       300
weighted avg       0.88      0.93      0.90       300

[[280   1]
 [ 19   0]]


In [73]:
print(df['is_returned'].value_counts())
print(df['is_returned'].value_counts(normalize=True) * 100)

is_returned
0    1405
1      95
Name: count, dtype: int64
is_returned
0    93.666667
1     6.333333
Name: proportion, dtype: float64


In [74]:
print("Antes:", pd.Series(y_train).value_counts().to_dict())
print("Después:", pd.Series(y_train_sm).value_counts().to_dict())

Antes: {0: 1124, 1: 76}
Después: {0: 1124, 1: 1124}


In [75]:
# Ver qué features importan para el modelo
importances = pd.Series(
    model.feature_importances_,
    index=features
).sort_values(ascending=False)

print(importances)

quantity               0.187409
shipping_method_enc    0.120208
payment_method_enc     0.110889
country_enc            0.084990
order_dayofweek        0.083378
order_month            0.079989
discount_pct           0.075464
category_enc           0.067651
unit_price             0.066961
age                    0.065275
is_discounted          0.057786
dtype: float64


Modelo predictivo 1: Predecir cuando un producto será devuelto o no:
El modelo no logró detectar patrones predictivos sobre devoluciones
(F1-score clase 1: 0.00). El análisis de feature importance muestra
distribución uniforme entre todas las variables (rango 0.05–0.18),
indicando ausencia de correlación entre las features disponibles y
el target is_returned.

Limitación del dataset: al ser datos sintéticos, order_status fue
asignado aleatoriamente sin dependencia de otras variables.


In [76]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

df_rated = df[df['rating'].notna()].copy()

X2 = df_rated[features]
y2 = df_rated['rating']

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42)

reg = RandomForestRegressor(n_estimators=100, random_state=42)
reg.fit(X2_train, y2_train)

y2_pred = reg.predict(X2_test)
print('MAE:', mean_absolute_error(y2_test, y2_pred))
print('R2 Score:', r2_score(y2_test, y2_pred))


MAE: 0.855
R2 Score: -0.10135898096632512
